In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
plt.rcParams.update({
    "font.family" : "serif",
    "font.size" : 15,
    "mathtext.fontset" : "stix",
    "font.serif" : ['STIXGeneral']
})

In [ ]:
folderVec = [
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_taylor_couette/25dx_noGeoUpdate/probes/probe",
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_taylor_couette/32dx_noGeoUpdate/probes/probe",
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_taylor_couette/40dx_noGeoUpdate/probes/probe",
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_taylor_couette/50dx_noGeoUpdate/probes/probe",
]

rLBVec = [
    [16, 41.6],
    [20, 52],
    [25, 65],
]

dOuterPhyVec = [
    0.26 * 2,
    0.26 * 2,
    0.26 * 2,
]

nuVec = [
    0.045,
    0.045,
    0.045,
]

omegaLBVec = [
    [0.00016666, 0.0],
    [0.00016666, 0.0],
    [0.00016666, 0.0],
]

avgStepRg = [
    [80000, 100000],
    [80000, 100000],
    [80000, 100000],
]

labelVec = [
    "Present LBM D3Q19 BGK D32",
    "Present LBM D3Q19 BGK D40",
    "Present LBM D3Q19 BGK D50",
    # "Present LBM D2Q9 BGK"
]

bdrIdx = [
    [31, 0, 32, 63], #! in, out, in, out
    [31, 0, 32, 63],
    [31, 0, 32, 63],
    [31, 0, 32, 63],
]


colorVec = [
    "#05A361",
    "#D33737",
    "#001AFF",
    '#FF5733',
    "#119100",
    "#B700FF",
]

markerVec = [
    'o',
    '^',
    's',
    "P",
    '*',
]

uLBMaxVec = []
for case in range(0, len(folderVec)):
    uLBMaxVec.append(omegaLBVec[case][0] * rLBVec[case][0])


In [ ]:
def read_col_probe(filePath, colIdx, skipHeader=2):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx)
        vec = vec[~np.isnan(vec)]
    f.close()
    return vec

def get_probe_coords(filePath):
    with open(filePath, 'r') as f:
        header = f.readline()
        coordsStr = header.split('(')[1].split(')')[0].split(',')
        x = float(coordsStr[0])
        y = float(coordsStr[1])
        z = float(coordsStr[2])
    return x,y,z

def read_csv_col(filePath, colIdx, skipHeader=1):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, delimiter=',', skip_header=skipHeader, usecols=colIdx)
    f.close()
    return vec


In [ ]:

rSurDVecVec = []
velVecVec = []
rhoNormVecVec = []

for case in range(0, len(folderVec)):
    files = sorted(glob.glob(os.path.join(folderVec[case], "probe*.txt")))
    nProbes = len(files)
    stepCol = read_col_probe(files[0], 0, 2)
    idxStart = int(np.abs(stepCol - avgStepRg[case][0]).argmin())
    idxEnd = int(np.abs(stepCol - avgStepRg[case][1]).argmin())

    rSurDVec = []
    velVec = []
    rhoNormVec = []

    rhoInnerLeft = read_col_probe(files[bdrIdx[case][0]], 2, 2)
    rhoOuterLeft = read_col_probe(files[bdrIdx[case][1]], 2, 2)
    rhoInnerRight = read_col_probe(files[bdrIdx[case][2]], 2, 2)
    rhoOuterRight = read_col_probe(files[bdrIdx[case][3]], 2, 2)
    avgRhoOuterLeft = np.mean(rhoOuterLeft[idxStart:idxEnd])
    avgRhoOuterRight = np.mean(rhoOuterRight[idxStart:idxEnd])
    avgRhoDiffLeft = avgRhoOuterLeft - np.mean(rhoInnerLeft[idxStart:idxEnd])
    avgRhoDiffRight = avgRhoOuterRight - np.mean(rhoInnerRight[idxStart:idxEnd])

    for iProbe in range(0, len(files)):
        x, y, z = get_probe_coords(files[iProbe])
        uy = read_col_probe(files[iProbe], 4, 2)
        meanUy = np.mean(uy[idxStart:idxEnd])
        rSurDVec.append(x / dOuterPhyVec[case])
        velVec.append(-meanUy / uLBMaxVec[case])

        rho = read_col_probe(files[iProbe], 2, 2)
        avgRho = np.mean(rho[idxStart:idxEnd])
        if x < 0:
            rhoNorm = (avgRho-avgRhoOuterLeft)/avgRhoDiffLeft
        else:
            rhoNorm = (avgRho-avgRhoOuterRight)/avgRhoDiffRight
        rhoNormVec.append(rhoNorm)


    rSurDVecVec.append(rSurDVec)
    velVecVec.append(velVec)
    rhoNormVecVec.append(rhoNormVec)



In [ ]:

#! analytical solution
case = 0
denom = rLBVec[case][1]**2 - rLBVec[case][0]**2
A = (omegaLBVec[case][1] * rLBVec[case][1]**2 - omegaLBVec[case][0] * rLBVec[case][0] **2) / denom
B = ((omegaLBVec[case][0] - omegaLBVec[case][1]) * rLBVec[case][0]**2 * rLBVec[case][1]**2) / denom
rVec = np.linspace(-rLBVec[case][1], rLBVec[case][1], 100, True)

##! azimuthal velocity (tangential velocity)
uRefVec = [-(A * r + B / r) / uLBMaxVec[case] for r in np.linspace(-rLBVec[case][1], rLBVec[case][1], 100, True)]
rSurDRefVec = rVec/(rLBVec[case][1] * 2)

idx = int(0.5*len(rSurDRefVec))
rSurDRef = np.insert(rSurDRefVec, idx, np.nan)
uRef = np.insert(uRefVec, idx, np.nan)

##! normalized pressure
def taylor_couette_pressure_diff(rO, rI, A, B, rho):
    term1 = 0.5 * A**2 * (rO**2 - rI**2)
    term2 = 2 * A * B * np.log(rO/rI)
    term3 = -0.5 * B**2 * (1/rO**2 - 1/rI**2)
    return rho * (term1 + term2 + term3)

refPreDiffInnerOuter = taylor_couette_pressure_diff(rLBVec[case][1], rLBVec[case][0], A, B, 1.0)
refPreDiffAnyROuter = []
for rr in range(0, len(rVec)):
    if rVec[rr] < 0:
        diffP =  taylor_couette_pressure_diff(rVec[rr], -rLBVec[case][1], A, B, 1.0)
    elif rVec[rr] > 0:
        diffP = taylor_couette_pressure_diff(rVec[rr], rLBVec[case][1], A, B, 1.0)
    else:
        diffP = np.nan
    refPreDiffAnyROuter.append(diffP / refPreDiffInnerOuter)

refPreDiffAnyROuter = np.insert(refPreDiffAnyROuter, idx, np.nan)



In [ ]:
figU, axU = plt.subplots(1, 1, figsize=(5, 5), facecolor='w', edgecolor='w')

axU.plot(rSurDRef, uRef, lw=1.5, c='k', label='Analytical solution')
for case in range(0, len(folderVec)):
    axU.plot(rSurDVecVec[case], velVecVec[case], marker=markerVec[case], ms=7, mew=1.5, fillstyle='none', markevery=2, c=colorVec[case], lw=0, label=labelVec[case])

axU.set_xlim(-0.5, 0.5)
axU.set_ylim(-1, 1)
# axU.legend(loc='center', frameon=False, fontsize=12)
axU.legend(loc='center', frameon=False, fontsize=13)
axU.set_xlabel(r"$r/(2R_{O})$")
axU.set_ylabel(r"$U_{t}/U_{t, max}$")

axU.xaxis.set_major_locator(MultipleLocator(0.25))
axU.xaxis.set_minor_locator(MultipleLocator(0.05))

axU.tick_params(width=1.5, axis='both', which='both', direction='in')
for spine in axU.spines.values():
    spine.set_linewidth(2.0)

In [ ]:
figP, axP = plt.subplots(1, 1, figsize=(5, 5), facecolor='w', edgecolor='w')

axP.plot(rSurDRef, refPreDiffAnyROuter, c='k', label='Analytical solution')
for case in range(0, len(folderVec)):
# for case in range(2, 3):
    axP.plot(rSurDVecVec[case], rhoNormVecVec[case], marker=markerVec[case], ms=7, mew=1.5, fillstyle='none', markevery=1, c=colorVec[case], lw=0, label=labelVec[case])
axP.set_xlim(-0.5, 0.5)
axP.set_ylim(-1.08, 0.08)
axP.legend(loc='upper center', frameon=False, fontsize=10)
# axP.legend(loc='upper center', frameon=False, fontsize=13)

axP.set_xlabel(r"$r/(2R_O)$")
axP.set_ylabel(r"$(P-P_O)/(P_O-P_I)$")

axP.yaxis.set_major_locator(MultipleLocator(0.2))
axP.yaxis.set_minor_locator(MultipleLocator(0.04))

axP.xaxis.set_major_locator(MultipleLocator(0.25))
axP.xaxis.set_minor_locator(MultipleLocator(0.05))

axP.tick_params(width=1.5, axis='both', which='both', direction='in')
for spine in axP.spines.values():
    spine.set_linewidth(2.0)